# NQ Futures — ORB Backtest v4 (Scale-Out)
### Using `backtesting.py`

**Trade lifecycle**
- **Entry** — 15:00 UTC close breaks above/below 14:00 UTC ORB close
- **Scale 1** — close 33% of position when unrealised profit >= 1× ATR14; move SL to breakeven
- **Scale 2** — close another 33% when unrealised profit >= 2× ATR14
- **Remainder** — final 34% runs to full TP at `risk_reward × ATR14`
- **SL** — initial SL = `sl_atr_mult × ATR14` from entry; moves to breakeven after Scale 1
- **No EOD force-close** — positions held until SL or TP is hit

| UTC | ET | SAST | Event |
|-----|----|------|-------|
| 13:00 | 08:00 | 15:00 | Pre-open bar — `prev_close` |
| 14:00 | 09:00 | 16:00 | ORB bar — trigger level |
| 15:00 | 10:00 | 17:00 | Entry bar |
| — | — | — | Scale 1 at +1× ATR → SL to BE |
| — | — | — | Scale 2 at +2× ATR |
| — | — | — | Remainder exits at TP |

## 1  Install dependencies

In [ ]:
!pip install yfinance backtesting pandas_ta -q


## 2  Download NQ data

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

CSV = 'NQ_1H_720D.csv'

try:
    import yfinance as yf
    raw = yf.download('NQ=F', period='720d', interval='1h',
                      auto_adjust=False, prepost=True, progress=False, threads=False)
    if raw.empty:
        raise ValueError('empty')
    raw.reset_index(inplace=True)
    raw.columns = ['Datetime','Open','High','Low','Close','Adj Close','Volume']
    raw.to_csv(CSV, index=False)
    print(f'Downloaded {len(raw):,} bars from yfinance')
except Exception as e:
    print(f'yfinance unavailable ({e}) — generating synthetic NQ data')
    np.random.seed(42)
    start = pd.Timestamp('2024-01-06')
    end   = pd.Timestamp('2026-01-10')
    idx = pd.date_range(start, end, freq='1h')
    idx = idx[(idx.hour >= 13) & (idx.hour <= 21)]
    n = len(idx)
    price = 18000.0
    closes = []
    for _ in range(n):
        price += np.random.normal(0, 40)
        price = max(price, 15000)
        closes.append(price)
    closes = np.array(closes)
    opens  = closes - np.random.uniform(-25, 25, n)
    highs  = np.maximum(opens, closes) + np.abs(np.random.normal(0, 80, n))
    lows   = np.minimum(opens, closes) - np.abs(np.random.normal(0, 80, n))
    vols   = np.random.randint(5000, 50000, n)
    raw = pd.DataFrame(dict(Datetime=idx, Open=opens, High=highs,
                            Low=lows, Close=closes, Volume=vols))
    raw.to_csv(CSV, index=False)
    print(f'Generated {n:,} synthetic bars')


## 3  Build ATR and attach to hourly data

ATR14 computed on daily bars, shifted 1 day to prevent lookahead, forward-filled onto hourly.

In [ ]:
import pandas_ta as ta

h = pd.read_csv(CSV, parse_dates=['Datetime'], index_col='Datetime')
h.index = pd.to_datetime(h.index, utc=True).tz_localize(None)
h = h[['Open','High','Low','Close','Volume']]
h = h.between_time('13:00', '21:00')
h = h.loc['2024-01-06':'2026-01-10']

# Daily OHLCV from session bars
session = h.between_time('14:00', '21:00')
daily = session.resample('1D').agg(
    Open  =('Open','first'),
    High  =('High','max'),
    Low   =('Low','min'),
    Close =('Close','last'),
    Volume=('Volume','sum')
).dropna()

daily['ATR14']    = ta.atr(daily['High'], daily['Low'], daily['Close'], length=14)
daily['ATR_pct']  = daily['ATR14'].rolling(60).apply(
    lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False
)
# Lag by 1 day — no lookahead
daily['ATR14_lag']   = daily['ATR14'].shift(1)
daily['ATR_pct_lag'] = daily['ATR_pct'].shift(1)

atr_hourly = daily[['ATR14_lag','ATR_pct_lag']].reindex(h.index, method='ffill')
h['ATR14']   = atr_hourly['ATR14_lag']
h['ATR_pct'] = atr_hourly['ATR_pct_lag']
h.dropna(inplace=True)

print(f'Hourly bars with ATR: {len(h):,}')
print(f'ATR14 range: {h["ATR14"].min():.0f} – {h["ATR14"].max():.0f} pts')
h.head()


## 4  Strategy — Scale-Out ORB

**How scale-out works in `backtesting.py`:**  
We open the full position at entry, then call `self.position.close(fraction)` at each scale level.  
SL is updated to breakeven by closing the original trade and re-entering — `backtesting.py` does not support  
modifying SL/TP on open trades directly, so we track breakeven manually and let price action handle it.

**Parameters:**

| Parameter | Default | Meaning |
|-----------|---------|--------|
| `risk_reward` | 3.0 | Final TP = RR × ATR14 from entry |
| `sl_atr_mult` | 1.0 | Initial SL = sl_atr_mult × ATR14 from entry |
| `entry_pct` | 0.1 | Min move past ORB close as fraction of ATR14 |
| `atr_pct_min` | 0.3 | Skip low-vol days below this ATR percentile |
| `skip_monday` | 0 | 1 = skip Mondays |
| `skip_friday` | 0 | 1 = skip Fridays |

In [ ]:
from backtesting import Backtest, Strategy

class ORB_ScaleOut(Strategy):
    """
    ORB with 3-stage scale-out. No EOD close — holds until SL or TP.

    Scale stages (tracked via _scale state):
      0 → full position open, watching for scale1 level
      1 → 33% closed at +1×ATR, SL moved to breakeven, watching for scale2
      2 → another 33% closed at +2×ATR, remainder running to TP
    """
    risk_reward  = 3.0
    sl_atr_mult  = 1.0
    entry_pct    = 0.1
    atr_pct_min  = 0.3
    skip_monday  = 0
    skip_friday  = 0

    def init(self):
        self.atr14   = self.I(lambda x: x, self.data.ATR14,   name='ATR14')
        self.atr_pct = self.I(lambda x: x, self.data.ATR_pct, name='ATR_pct')
        self._reset_day(None)
        # Scale tracking — persists across days while trade is open
        self._scale       = 0      # 0=none, 1=scale1 done, 2=scale2 done
        self._entry_price = None
        self._direction   = None   # +1 long, -1 short
        self._atr_at_entry= None
        self._scale1_lvl  = None   # price level for scale 1
        self._scale2_lvl  = None   # price level for scale 2
        self._be_sl       = None   # breakeven SL level

    def _reset_day(self, date):
        self._day          = date
        self._orb_close    = None
        self._atr          = None
        self._range_ready  = False
        self._traded_today = False

    def _reset_trade(self):
        self._scale        = 0
        self._entry_price  = None
        self._direction    = None
        self._atr_at_entry = None
        self._scale1_lvl   = None
        self._scale2_lvl   = None
        self._be_sl        = None

    def next(self):
        t     = self.data.index[-1]
        date  = t.date()
        hour  = t.hour
        close = self.data.Close[-1]
        atr   = self.data.ATR14[-1]
        apct  = self.data.ATR_pct[-1]

        # ── New day reset ─────────────────────────────────────────────────
        if date != self._day:
            self._reset_day(date)

        # ── Manage open position: scale-out and breakeven SL ──────────────
        if self.position:
            direction = self._direction
            price     = close

            # Scale 1 — close 33% at +1× ATR, move SL to breakeven
            if self._scale == 0:
                if (direction == 1  and price >= self._scale1_lvl) or \
                   (direction == -1 and price <= self._scale1_lvl):
                    self.position.close(0.333)
                    self._scale  = 1
                    self._be_sl  = self._entry_price

            # Scale 2 — close 50% of remaining (=33% of original) at +2× ATR
            elif self._scale == 1:
                if (direction == 1  and price >= self._scale2_lvl) or \
                   (direction == -1 and price <= self._scale2_lvl):
                    self.position.close(0.5)
                    self._scale = 2

            # Breakeven SL enforcement (manual — backtesting.py can't modify SL)
            # After scale 1, if price retraces to entry, close the remainder
            if self._scale >= 1 and self._be_sl is not None:
                if (direction == 1  and price <= self._be_sl) or \
                   (direction == -1 and price >= self._be_sl):
                    self.position.close()
                    self._reset_trade()
                    return

        else:
            # No open position — reset scale state
            if self._scale != 0:
                self._reset_trade()

        # ── 14:00 UTC — ORB bar ───────────────────────────────────────────
        if hour == 14:
            dow = t.weekday()
            if self.skip_monday and dow == 0: return
            if self.skip_friday and dow == 4: return
            if apct < self.atr_pct_min:       return
            self._orb_close   = close
            self._atr         = atr
            self._range_ready = True
            return

        # ── 15:00 UTC — sole entry bar ────────────────────────────────────
        if hour == 15:
            if not self._range_ready:  return
            if self._traded_today:     return
            if self.position:          return

            orb     = self._orb_close
            atr_val = self._atr
            thresh  = self.entry_pct  * atr_val
            sl_dist = self.sl_atr_mult * atr_val
            tp_dist = self.risk_reward * atr_val

            if close > orb + thresh:       # LONG
                self.buy(
                    sl = close - sl_dist,
                    tp = close + tp_dist,
                )
                self._entry_price  = close
                self._direction    = 1
                self._atr_at_entry = atr_val
                self._scale1_lvl   = close + 1.0 * atr_val
                self._scale2_lvl   = close + 2.0 * atr_val
                self._scale        = 0
                self._traded_today = True

            elif close < orb - thresh:     # SHORT
                self.sell(
                    sl = close + sl_dist,
                    tp = close - tp_dist,
                )
                self._entry_price  = close
                self._direction    = -1
                self._atr_at_entry = atr_val
                self._scale1_lvl   = close - 1.0 * atr_val
                self._scale2_lvl   = close - 2.0 * atr_val
                self._scale        = 0
                self._traded_today = True


## 5  Baseline run

In [ ]:
bt = Backtest(
    h,
    ORB_ScaleOut,
    cash           = 100_000,
    commission     = 0.0002,
    trade_on_close = True,
)

stats = bt.run()
pd.set_option('display.max_rows', None)
print(stats)


## 6  Interactive chart

In [ ]:
bt.plot()


## 7  Optimise

In [ ]:
opt_stats, heatmap = bt.optimize(
    risk_reward  = [2.0, 2.5, 3.0, 4.0],
    sl_atr_mult  = [0.5, 0.75, 1.0, 1.25],
    entry_pct    = [0.0, 0.05, 0.1, 0.15],
    atr_pct_min  = [0.0, 0.2, 0.3, 0.4],
    skip_monday  = [0, 1],
    skip_friday  = [0, 1],
    maximize     = 'Sharpe Ratio',
    constraint   = lambda p: p.risk_reward > p.sl_atr_mult,
    return_heatmap = True,
)

print('=== Best parameters ===')
print(f"  risk_reward  : {opt_stats._strategy.risk_reward}")
print(f"  sl_atr_mult  : {opt_stats._strategy.sl_atr_mult}")
print(f"  entry_pct    : {opt_stats._strategy.entry_pct}")
print(f"  atr_pct_min  : {opt_stats._strategy.atr_pct_min}")
print(f"  skip_monday  : {bool(opt_stats._strategy.skip_monday)}")
print(f"  skip_friday  : {bool(opt_stats._strategy.skip_friday)}")
print()
pd.set_option('display.max_rows', None)
print(opt_stats)


## 8  Best-params run + trade breakdown

In [ ]:
bt_best = Backtest(
    h, ORB_ScaleOut,
    cash           = 100_000,
    commission     = 0.0002,
    trade_on_close = True,
)

best = bt_best.run(
    risk_reward = opt_stats._strategy.risk_reward,
    sl_atr_mult = opt_stats._strategy.sl_atr_mult,
    entry_pct   = opt_stats._strategy.entry_pct,
    atr_pct_min = opt_stats._strategy.atr_pct_min,
    skip_monday = opt_stats._strategy.skip_monday,
    skip_friday = opt_stats._strategy.skip_friday,
)

trades = best['_trades'].copy()
trades['EntryTime'] = pd.to_datetime(trades['EntryTime'])
trades['ExitTime']  = pd.to_datetime(trades['ExitTime'])
trades['PnL ($)']   = trades['PnL'].round(2)
trades['ReturnPct'] = (trades['ReturnPct'] * 100).round(2)
trades['Duration']  = trades['ExitTime'] - trades['EntryTime']
trades['Entry_SAST']= trades['EntryTime'] + pd.Timedelta(hours=2)
trades['Exit_SAST'] = trades['ExitTime']  + pd.Timedelta(hours=2)

print('=== Best-params stats ===')
pd.set_option('display.max_rows', None)
print(best)
print()
print('=== Scale-out sanity ===')
print(f"Entry hours UTC : {sorted(trades['EntryTime'].dt.hour.unique())}  ← should be [15]")
print(f"Total trade legs: {len(trades)}  (3 legs per full trade = {len(trades)//3} full trades)")
print()
trades[['Entry_SAST','Exit_SAST','Duration','EntryPrice','ExitPrice','PnL ($)','ReturnPct','Size']].head(30)


## 9  Day-of-week breakdown

In [ ]:
import matplotlib.pyplot as plt

trades['DOW'] = trades['EntryTime'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday']

dow = trades.groupby('DOW').agg(
    Legs     = ('PnL ($)', 'count'),
    WinRate  = ('PnL ($)', lambda x: (x > 0).mean()),
    AvgPnL   = ('PnL ($)', 'mean'),
    TotalPnL = ('PnL ($)', 'sum'),
).reindex(dow_order).round(2)

print(dow)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in dow['AvgPnL']]
dow['AvgPnL'].plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Avg PnL per leg by day of week')
axes[0].axhline(0, color='black', linewidth=0.8)

dow['WinRate'].plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Win rate by day of week')
axes[1].axhline(0.5, color='red', linestyle='--', label='50%')
axes[1].legend()

plt.tight_layout()
plt.show()


## 10  Push notebook to GitHub

Uploads this notebook to your repository:  
`hlenganeindustries-alt/Nasdaq-Futures-backtesting`

You will be prompted to enter your **GitHub Personal Access Token**.  
Generate one at: https://github.com/settings/tokens → *Generate new token (classic)* → tick `repo` scope.

In [ ]:
import requests, base64, json, os
from getpass import getpass

# ── Config ────────────────────────────────────────────────────────────────────
REPO   = 'hlenganeindustries-alt/Nasdaq-Futures-backtesting'
BRANCH = 'main'
PATH   = 'NQ_ORB_v4.ipynb'   # path inside the repo

# ── Token input (hidden) ──────────────────────────────────────────────────────
TOKEN = getpass('Enter your GitHub Personal Access Token: ')

# ── Read this notebook file ───────────────────────────────────────────────────
notebook_path = '/content/NQ_ORB_v4.ipynb'
if not os.path.exists(notebook_path):
    # Colab sometimes saves under a different name — find it
    candidates = [f for f in os.listdir('/content') if f.endswith('.ipynb')]
    if candidates:
        notebook_path = f'/content/{candidates[0]}'
        print(f'Using notebook: {notebook_path}')
    else:
        raise FileNotFoundError('No .ipynb found in /content — save the notebook first (Ctrl+S)')

with open(notebook_path, 'rb') as f:
    content_b64 = base64.b64encode(f.read()).decode()

# ── GitHub API ────────────────────────────────────────────────────────────────
headers = {
    'Authorization': f'token {TOKEN}',
    'Accept'       : 'application/vnd.github.v3+json',
}
api_url = f'https://api.github.com/repos/{REPO}/contents/{PATH}'

# Check if file already exists (need its SHA to update)
existing = requests.get(api_url, headers=headers, params={'ref': BRANCH})
sha = existing.json().get('sha') if existing.status_code == 200 else None

# Commit message
commit_msg = 'Update NQ_ORB_v4 — scale-out strategy with ATR-based SL/TP'

payload = {
    'message': commit_msg,
    'content': content_b64,
    'branch' : BRANCH,
}
if sha:
    payload['sha'] = sha   # required for updates

response = requests.put(api_url, headers=headers, data=json.dumps(payload))

# ── Result ────────────────────────────────────────────────────────────────────
if response.status_code in (200, 201):
    action = 'Updated' if sha else 'Created'
    html_url = response.json()['content']['html_url']
    print(f'✓ {action} successfully!')
    print(f'  {html_url}')
else:
    print(f'✗ Failed ({response.status_code})')
    print(response.json().get('message', response.text))
